# DATE2 实验3：朴素预取干扰与MemDomain协同优化

论文只报告一个最终`MemDomain`；内部诊断候选不会进入论文图表。


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve().parent if Path.cwd().name=='fig' else Path.cwd().resolve()
OUT=ROOT/'outputs/DATE2';FIG=ROOT/'fig/DATE2';FIG.mkdir(parents=True,exist_ok=True)
PUBLIC=['Static-NoPF','Static-NaivePF','Dynamic-NoPF','Dynamic-NaivePF','MemDomain']
FINAL_INTERNAL='MemDomain-'+'Safe'
def public_rows(frame):
    q=frame[frame.baseline.isin(['Static-NoPF','Static-NaivePF','Dynamic-NoPF',
                                 FINAL_INTERNAL])].copy()
    q['baseline']=q.baseline.replace({FINAL_INTERNAL:'MemDomain'})
    assert set(q.baseline)==set(PUBLIC)
    return q

rows=[]
for path in sorted((OUT/'window_chunk').glob('w*_c*.csv')):
    w,c=map(int,re.search(r'w(\d+)_c(\d+)',path.stem).groups())
    q=public_rows(pd.read_csv(path));q['window']=w;q['chunk_tiles']=c;rows.append(q)
d=pd.concat(rows,ignore_index=True)
assert d.groupby(['window','chunk_tiles']).size().eq(5).all()


In [ ]:
wide=d.pivot_table(index=['window','chunk_tiles'],columns='baseline',values='total_cycles')
wide['naive_change_pct']=(wide['Static-NaivePF']/wide['Static-NoPF']-1)*100
wide['memdomain_gain_pct']=(1-wide['MemDomain']/wide['Dynamic-NaivePF'])*100
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,col,title in zip(axes,['naive_change_pct','memdomain_gain_pct'],
                       ['Naive prefetch change','MemDomain gain over matched dynamic prefetch']):
    p=wide[col].unstack();im=ax.imshow(p,aspect='auto',cmap='RdYlGn')
    ax.set_xticks(range(len(p.columns)),p.columns);ax.set_yticks(range(len(p.index)),p.index)
    ax.set_xlabel('Chunk tiles');ax.set_ylabel('Prefetch window');ax.set_title(title)
    fig.colorbar(im,ax=ax,label='%')
plt.tight_layout();plt.savefig(FIG/'exp3_public_prefetch.pdf',bbox_inches='tight');plt.show()
display(wide.reset_index())


正值`MemDomain gain`表示在相同预取条件下统一Bank池和动态映射降低周期；所有结论从重跑CSV计算。
